# 08_03 Cross-article promotion features

## Mechanism and expected effect (stated before running)

The model knows an article's **own** promotion schedule (`action_on_forecast_day`,
`action_during_horizon`) but is blind to what the *rest of the counter* is doing.
Two mechanisms make that visible in demand:

1. **Substitution (cannibalization).** When other articles of the same
   Warenklasse (e.g. another Schweinefleisch cut) are promoted at the store,
   an unpromoted article's demand should fall — shoppers switch to the
   discounted like-for-like item.
2. **Flight intensity.** Days with many simultaneous actions are ad-flight days
   with different store traffic; both promoted and unpromoted articles see a
   regime the model cannot currently separate from ordinary days.

**Baseline and dataset.** This batch runs on `data/processed/transactions_fixed`
— the per-Bundesland-calendar dataset that `07_01` established as the one to
carry forward — with `DATA_DIR` flipped accordingly, so the design, the feature
build, and the refit all use the corrected data. The baseline is the published
`run_fixed_data` two-stage fit (seed 42, identical config): row 57.44%,
bias −1.21%, article-store-week 37.12%, article-day 23.96%. The `07_02` seed
floor (row sd 0.053 pp, bias sd 0.19 pp) was measured on this same dataset and
configuration, so it applies without a transfer assumption.

**Pre-registered expectation.** The mechanism check below holds the population
fixed: the primary tables use only series with no-own-action rows in **all
three** cross-action buckets within the fixed 140-calendar-day evaluation
window (9,390 of 21,922 series), so bucket differences reflect the regime and
not composition. On that fixed population the model **overforecasts quiet-class
days by +11.4%** (+10.5% weekday-controlled) against +4.6% with 1–2 competing
class actions and +4.1% with 3+, and the same series' demand on competing-action
days is 0.78×/0.71× of its quiet-day weekday baseline. An earlier measurement of
this batch on the superseded `transactions` dataset (preserved in
`backup_08_03_old_data/`) saw a +20.7% quiet-day bias — about half of that
signal was the calendar defect itself (NW stores recorded active with zero
sales on NW-only holidays land in the quiet buckets), which the fixed data
removes; the demand-level substitution effect is unchanged. The honest budget
is therefore **~0.05–0.2 pp row WAPE**, and — because the old-data run bought
its −0.20 pp partly with a −0.66 pp pooled-bias shift (a guardrail violation) —
**pooled bias is an explicit pass/fail criterion for this run**. Following the
`06_02` channel lesson, the *share* variant is expected to be the most usable
form for the trees.

In [1]:
from pathlib import Path
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.benchmark import load_benchmark_design
from src.models.lightgbm import TWO_STAGE_MODEL_NAME
from src.models.results import result_path

pd.set_option('display.max_columns', 40)

design = load_benchmark_design()
BASELINE_FORECASTS = (ROOT / 'reports' / 'results' / 'run_fixed_data' / 'forecasts'
                      / 'artikel_markt_multi7days_lightgbm_two_stage.csv')
NEW_FORECASTS = result_path(TWO_STAGE_MODEL_NAME, design)

con = duckdb.connect()
con.execute('PRAGMA threads=4')
con.execute('''
    CREATE OR REPLACE TEMP TABLE base AS
    SELECT ARTIKEL_ID::BIGINT AS ARTIKEL_ID, MARKT_ID::BIGINT AS MARKT_ID,
           CAST(origin AS DATE) AS origin, CAST(period AS DATE) AS period,
           actual::DOUBLE AS actual, forecast::DOUBLE AS forecast,
           action_on_forecast_day::INTEGER AS own_action,
           EXTRACT(ISODOW FROM CAST(period AS DATE)) AS weekday
    FROM read_csv_auto(?) WHERE is_active
''', [str(BASELINE_FORECASTS)])
print(con.execute('SELECT COUNT(*), COUNT(DISTINCT origin) FROM base').fetchone())

(2498967, 20)


## Mechanism verification on the pre-batch forecasts

Cross-action context is reconstructed from the processed transactions: for every
store-day, the number of *other* articles of the same Warenklasse
(`N_WARENKLASSE_KBEZ`) with `AKTION_KENNZEICHEN = 1`, and the store-wide count.

**Fixed population, fixed calendar days.** All rows come from the same fixed
140 target dates of the 20 evaluation origins, and the primary tables restrict
to the series (article × store) that have no-own-action rows in **all three**
cross-action buckets inside that window — every bucket is measured on the same
series over the same calendar span. The unrestricted (mixed-population) table
is reported once for coverage. The tables condition on rows **without** own
action (the cannibalization case), a weekday-controlled variant (Tue–Fri), and
a within-series demand comparison against quiet-day weekday baselines.

In [2]:
con.execute('''
    CREATE OR REPLACE TEMP TABLE txn AS
    SELECT ARTIKEL_ID::BIGINT AS ARTIKEL_ID, MARKT_ID::BIGINT AS MARKT_ID,
           CAST(DATE AS DATE) AS period,
           N_WARENKLASSE_KBEZ AS warenklasse,
           (COALESCE(AKTION_KENNZEICHEN, 0) = 1)::INTEGER AS action_flag
    FROM read_parquet(?)
    WHERE (is_fcm OR is_pseudo) AND WGR_ID IN (890, 900)
''', [str(design.data_dir / '*.parquet')])
con.execute('''
    CREATE OR REPLACE TEMP TABLE joined AS
    WITH art_wk AS (
        SELECT ARTIKEL_ID, ANY_VALUE(warenklasse) AS warenklasse FROM txn GROUP BY 1
    ),
    wk_actions AS (
        SELECT MARKT_ID, period, warenklasse,
               COUNT(DISTINCT ARTIKEL_ID) FILTER (WHERE action_flag = 1) AS n_action
        FROM txn GROUP BY 1, 2, 3
    ),
    store_actions AS (
        SELECT MARKT_ID, period,
               COUNT(DISTINCT ARTIKEL_ID) FILTER (WHERE action_flag = 1) AS n_store
        FROM txn GROUP BY 1, 2
    )
    SELECT base.*, art_wk.warenklasse,
           COALESCE(w.n_action, 0) - base.own_action AS cross_class_actions,
           COALESCE(s.n_store, 0) - base.own_action AS cross_store_actions
    FROM base
    JOIN art_wk USING (ARTIKEL_ID)
    LEFT JOIN wk_actions AS w
      ON base.MARKT_ID = w.MARKT_ID AND base.period = w.period
         AND art_wk.warenklasse = w.warenklasse
    LEFT JOIN store_actions AS s
      ON base.MARKT_ID = s.MARKT_ID AND base.period = s.period
''')

BUCKET = ("CASE WHEN cross_class_actions <= 0 THEN '0' "
          "WHEN cross_class_actions <= 2 THEN '1-2' ELSE '3+' END")
# Fixed population: series whose no-own-action rows visit all three buckets
# within the fixed evaluation window.
con.execute(f'''
    CREATE OR REPLACE TEMP TABLE fixed_pop AS
    SELECT ARTIKEL_ID, MARKT_ID FROM joined WHERE own_action = 0
    GROUP BY 1, 2
    HAVING SUM((cross_class_actions = 0)::INTEGER) > 0
       AND SUM((cross_class_actions BETWEEN 1 AND 2)::INTEGER) > 0
       AND SUM((cross_class_actions >= 3)::INTEGER) > 0
''')
coverage = con.execute('''
    SELECT (SELECT COUNT(*) FROM fixed_pop) AS fixed_series,
           COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID)) AS all_series,
           SUM(actual) FILTER (WHERE (ARTIKEL_ID, MARKT_ID) IN
               (SELECT (ARTIKEL_ID, MARKT_ID) FROM fixed_pop)) / SUM(actual)
               AS volume_share
    FROM joined WHERE own_action = 0
''').fetchdf()
display(coverage.style.format({'fixed_series': '{:,.0f}', 'all_series': '{:,.0f}',
                               'volume_share': '{:.1%}'}))

for label, extra, pop in [
    ('no own action — FIXED population', 'own_action = 0', True),
    ('no own action, Tue-Fri — FIXED population',
     'own_action = 0 AND weekday BETWEEN 2 AND 5', True),
    ('no own action — mixed population (coverage)', 'own_action = 0', False),
    ('own action — mixed population', 'own_action = 1', False),
]:
    join = 'JOIN fixed_pop USING (ARTIKEL_ID, MARKT_ID)' if pop else ''
    table = con.execute(f'''
        SELECT {BUCKET} AS cross_class_bucket,
               COUNT(*) AS rows_n, SUM(actual) AS volume,
               SUM(forecast - actual) / SUM(actual) AS rel_bias,
               SUM(ABS(forecast - actual)) / SUM(actual) AS wape
        FROM joined {join} WHERE {extra} GROUP BY 1 ORDER BY 1
    ''').fetchdf()
    print(f'--- {label} ---')
    display(table.style.format({'rows_n': '{:,.0f}', 'volume': '{:,.0f}',
                                'rel_bias': '{:+.2%}', 'wape': '{:.2%}'}))

demand_check = con.execute(f'''
    WITH quiet AS (
        SELECT ARTIKEL_ID, MARKT_ID, weekday, AVG(actual) AS mu
        FROM joined JOIN fixed_pop USING (ARTIKEL_ID, MARKT_ID)
        WHERE own_action = 0 AND cross_class_actions = 0
        GROUP BY 1, 2, 3
    )
    SELECT {BUCKET} AS cross_class_bucket,
           SUM(joined.actual) / SUM(quiet.mu) AS demand_vs_quiet_weekday_baseline,
           COUNT(*) AS rows_n
    FROM joined
    JOIN fixed_pop USING (ARTIKEL_ID, MARKT_ID)
    JOIN quiet USING (ARTIKEL_ID, MARKT_ID, weekday)
    WHERE own_action = 0 AND quiet.mu > 0
    GROUP BY 1 ORDER BY 1
''').fetchdf()
display(demand_check.style.format({'demand_vs_quiet_weekday_baseline': '{:.3f}',
                                   'rows_n': '{:,.0f}'}))

,fixed_series,all_series,volume_share
0,"9,390","21,922",29.9%


--- no own action — FIXED population ---


,cross_class_bucket,rows_n,volume,rel_bias,wape
0,0,"93,927","21,794",+11.40%,96.61%
1,1-2,"357,638","134,030",+4.61%,82.81%
2,3+,"499,387","205,021",+4.06%,90.79%


--- no own action, Tue-Fri — FIXED population ---


,cross_class_bucket,rows_n,volume,rel_bias,wape
0,0,"62,937","13,666",+10.47%,97.00%
1,1-2,"241,905","82,622",+5.29%,85.61%
2,3+,"331,743","118,788",+5.74%,95.73%


--- no own action — mixed population (coverage) ---


,cross_class_bucket,rows_n,volume,rel_bias,wape
0,0,"406,355","78,932",+2.17%,96.11%
1,1-2,"476,152","177,062",+3.56%,82.57%
2,3+,"1,326,905","952,406",-0.32%,70.06%


--- own action — mixed population ---


,cross_class_bucket,rows_n,volume,rel_bias,wape
0,0,"31,414","56,292",+4.39%,54.06%
1,1-2,"72,914","214,824",+0.50%,48.44%
2,3+,"185,227","1,182,078",-3.44%,42.72%


,cross_class_bucket,demand_vs_quiet_weekday_baseline,rows_n
0,0,1.000,"50,228"
1,1-2,0.777,"91,536"
2,3+,0.710,"67,047"


**Reading the verification.** On the fixed population — the same series over
the same fixed calendar days in every bucket — the gradient is +11.4% (quiet) →
+4.6% (1–2) → +4.1% (3+), robust to weekday control (+10.5%/+5.3%/+5.7%), and
within the same series and weekday, demand on competing-action days is 0.78×
(1–2) and 0.71× (3+) of the quiet-day baseline. Compared to the superseded
dataset the *residual* gradient shrank (the calendar defect accounted for
roughly half of it) while the *demand* dip is unchanged — the substitution
mechanism is real, and what remains uncaptured by the model is worth single-digit
percent bias on a small-volume slice.

## The features

Two columns join the contract (`FEATURE_BUILDER_VERSION` 2026-08-07.4 →
2026-08-16.2, all partitions rebuilt on `transactions_fixed`, only the
two-stage model refit; both stages receive both):

- **`same_class_other_actions_on_forecast_day`** — number of *other* articles of
  the same Warenklasse promoted at the store on the target date. The counts come
  from the same known-ahead promotion schedule as `action_on_forecast_day`.
- **`same_class_action_share_on_forecast_day`** — that count divided by the
  number of distinct class articles with at least one active row at the store in
  [origin − 28 d, origin) — the denominator uses only rows strictly before the
  origin.

A third candidate, a store-wide flight-intensity count
(`store_other_actions_on_forecast_day`), was implemented, measured on the
superseded dataset, and **dropped before this run**: it was the weakest of the
three mechanisms, and on the old-data run it was the dominant user of the group
(occurrence rank 14.8/55, 10× the gain share of both class features combined)
while pooled bias shifted −0.97% → −1.63% — the identified guardrail violation.
It is recorded in `REMOVED_FEATURE_COLUMNS` so stale caches are rejected. With
it gone, this run is a clean test of the substitution mechanism alone.

The Warenklasse mapping (`N_WARENKLASSE_KBEZ`, 18 classes; largest:
Schweinefleisch 137 articles, Rindfleisch 81, Rohwurst Stückware 46) is a static
article attribute read from the processed transactions; unmapped articles
degrade to zero counts and a missing share. Unit-tested: counts on the action
day, own-action subtraction, unmapped degradation, and that the assortment
denominator ignores articles first seen after the origin (91 tests pass).

## Results

Both runs cover the same 20 origins on `transactions_fixed`; the baseline is
the published `run_fixed_data` two-stage fit (seed 42, identical configuration
and design apart from the two new features).

In [3]:
con.execute('''
    CREATE OR REPLACE TEMP TABLE new_run AS
    SELECT ARTIKEL_ID::BIGINT AS ARTIKEL_ID, MARKT_ID::BIGINT AS MARKT_ID,
           CAST(origin AS DATE) AS origin, CAST(period AS DATE) AS period,
           actual::DOUBLE AS actual, forecast::DOUBLE AS forecast
    FROM read_csv_auto(?) WHERE is_active
''', [str(NEW_FORECASTS)])

GRAINS = {
    'row': None,
    'article-store-week': 'ARTIKEL_ID, MARKT_ID, origin',
    'article-day': 'ARTIKEL_ID, origin, period',
}


def score(table):
    out = {'run': table}
    for grain, keys in GRAINS.items():
        inner = (f'SELECT actual, forecast FROM {table}' if keys is None else
                 f'SELECT SUM(actual) AS actual, SUM(forecast) AS forecast '
                 f'FROM {table} GROUP BY {keys}')
        wape, bias = con.execute(
            f'SELECT SUM(ABS(forecast - actual)) / SUM(actual), '
            f'SUM(forecast - actual) / SUM(actual) FROM ({inner})').fetchone()
        out[grain] = wape
        if grain == 'row':
            out['bias'] = bias
    return out


comparison = pd.DataFrame([score('base'), score('new_run')])
comparison['run'] = ['baseline (run_fixed_data)', 'with cross-action features']
for g in GRAINS:
    comparison[f'Δ {g} (pp)'] = (comparison[g] - comparison[g].iloc[0]) * 100
display(comparison.style.format({**{g: '{:.2%}' for g in GRAINS}, 'bias': '{:+.2%}',
                                 **{f'Δ {g} (pp)': '{:+.2f}' for g in GRAINS}}))

,run,row,bias,article-store-week,article-day,Δ row (pp),Δ article-store-week (pp),Δ article-day (pp)
0,baseline (run_fixed_data),57.44%,-1.21%,37.12%,23.96%,+0.00,+0.00,+0.00
1,with cross-action features,57.61%,-1.16%,37.35%,24.15%,+0.17,+0.23,+0.20


In [4]:
per_origin = con.execute('''
    SELECT new_run.origin,
           SUM(ABS(base.forecast - base.actual)) / SUM(base.actual) AS wape_base,
           SUM(ABS(new_run.forecast - new_run.actual)) / SUM(new_run.actual) AS wape_new
    FROM new_run JOIN base USING (ARTIKEL_ID, MARKT_ID, origin, period)
    GROUP BY 1 ORDER BY 1
''').fetchdf()
per_origin['delta_pp'] = (per_origin.wape_new - per_origin.wape_base) * 100
print(f'origins improved: {(per_origin.delta_pp < 0).sum()} / {len(per_origin)}')
display(per_origin.style.format({'wape_base': '{:.2%}', 'wape_new': '{:.2%}',
                                 'delta_pp': '{:+.2f}'}))

origins improved: 9 / 20


,origin,wape_base,wape_new,delta_pp
0,2026-03-02 00:00:00,55.63%,55.29%,-0.35
1,2026-03-09 00:00:00,60.93%,60.66%,-0.27
2,2026-03-16 00:00:00,57.49%,57.92%,+0.43
3,2026-03-23 00:00:00,62.05%,62.27%,+0.22
4,2026-03-30 00:00:00,48.71%,48.94%,+0.24
5,2026-04-06 00:00:00,58.53%,58.52%,-0.02
6,2026-04-13 00:00:00,67.63%,67.54%,-0.09
7,2026-04-20 00:00:00,58.43%,59.09%,+0.66
8,2026-04-27 00:00:00,54.61%,54.84%,+0.22
9,2026-05-04 00:00:00,56.68%,56.65%,-0.03


In [5]:
# Did the bias gradient close? Same fixed population and buckets as the
# verification, scored with the new forecasts.
closed = con.execute(f'''
    SELECT {BUCKET} AS cross_class_bucket,
           SUM(joined.forecast - joined.actual) / SUM(joined.actual) AS bias_base,
           SUM(new_run.forecast - new_run.actual) / SUM(new_run.actual) AS bias_new,
           SUM(ABS(joined.forecast - joined.actual)) / SUM(joined.actual) AS wape_base,
           SUM(ABS(new_run.forecast - new_run.actual)) / SUM(new_run.actual) AS wape_new
    FROM joined
    JOIN fixed_pop USING (ARTIKEL_ID, MARKT_ID)
    JOIN new_run USING (ARTIKEL_ID, MARKT_ID, origin, period)
    WHERE joined.own_action = 0
    GROUP BY 1 ORDER BY 1
''').fetchdf()
display(closed.style.format({'bias_base': '{:+.2%}', 'bias_new': '{:+.2%}',
                             'wape_base': '{:.2%}', 'wape_new': '{:.2%}'}))

importance = pd.read_csv(
    ROOT / 'reports' / 'results' / 'feature_importance'
    / 'artikel_markt_multi7days_lightgbm_two_stage.csv')
new_features = ['same_class_other_actions_on_forecast_day',
                'same_class_action_share_on_forecast_day']
importance['rank'] = importance.groupby(['evaluation_origin', 'stage'])['gain']     .rank(ascending=False)
summary = (importance[importance.feature.isin(new_features)]
           .groupby(['stage', 'feature'])
           .agg(mean_rank=('rank', 'mean'), mean_gain_share=('gain_share', 'mean'))
           .reset_index())
n_features = importance.groupby(['evaluation_origin', 'stage'])['feature'].nunique().max()
print(f'features per stage: {n_features}')
display(summary.style.format({'mean_rank': '{:.1f}', 'mean_gain_share': '{:.3%}'}))

,cross_class_bucket,bias_base,bias_new,wape_base,wape_new
0,0,+11.40%,+11.23%,96.61%,96.31%
1,1-2,+4.61%,+5.13%,82.81%,83.12%
2,3+,+4.06%,+4.05%,90.79%,90.98%


features per stage: 54


,stage,feature,mean_rank,mean_gain_share
0,occurrence,same_class_action_share_on_forecast_day,31.8,0.049%
1,occurrence,same_class_other_actions_on_forecast_day,37.8,0.024%
2,positive_quantity,same_class_action_share_on_forecast_day,50.6,0.003%
3,positive_quantity,same_class_other_actions_on_forecast_day,50.6,0.003%


## Verdict

**Null on the corrected data — the trees barely use the features, the targeted
miscalibration does not close, and pooled WAPE moves slightly the wrong way.
Recommendation: revert the two class features; keep this notebook as the
record.**

Measured against the `07_02` seed floor (row sd 0.053 pp, bias sd 0.19 pp),
which was estimated on exactly this dataset and configuration:

- Row WAPE 57.44% → 57.61% (**+0.17 pp**, ≈ 3 sd), article-store-week
  +0.23 pp, article-day +0.20 pp; only 9/20 origins improved. Bias
  −1.21% → −1.16% is unchanged (0.3 sd). The pre-registered budget was
  +0.05–0.2 pp of *gain*; the run delivered a small but probably real cost.
  (The seed floor was measured for a fixed feature set; adding columns also
  perturbs the column-sampling randomness, so part of the +0.17 pp may be
  fit lottery — but nothing in this run argues there is a gain hiding under
  it.)
- **The pre-registered signature did not appear**: the fixed-population
  quiet-class bias moved +11.40% → +11.23% — no closure — in sharp contrast
  to the superseded-data run, where the (then three-feature) batch closed
  +20.7% → +6.8%. That old closure evidently rode on the
  flight-intensity/data-defect channels that the fixed data and the dropped
  store count removed.
- **Importance confirms the non-adoption**: `same_class_action_share` ranks
  31.8/54 in the occurrence stage at 0.049% gain share (the share form did
  again beat the raw count, 0.049% vs 0.024%), and the quantity stage ignores
  both (~0.003%).

Why a real demand mechanism (the 0.78×/0.71× same-series dip survives on fixed
data) fails as a feature here: the *residual* miscalibration left after the
data fix is +11% bias on a slice carrying ~0.9% of volume — a theoretical
ceiling of ≈ 0.06 pp row WAPE, at or below the noise floor. The model's
existing features already price most of the competition regime; what the check
measured as demand-level substitution is largely absorbed through
`rolling_*`/`demand_rate_*` histories that co-move with promotion cadence.
This is the `05_05`-familiar outcome: mechanism real, remaining exploitable
signal too small.

**If the mechanism is revisited, it should be through intensity, not breadth**:
depth-weighted competition from the raw campaign data (`AKTIONSNUMMER` +
`UMS_VK_PREIS`; median measured discount depth 21.7%, p90 38%) concentrates
the signal on the deep-discount events that plausibly drive the dip, instead
of counting token actions equally.